In [14]:
from google.colab import auth
from google.cloud import bigquery
import pandas as pd
import numpy as np

print("Starting Google Authentication...")
auth.authenticate_user()

PROJECT_ID = "glp-1-513513"

client = bigquery.Client(project=PROJECT_ID)
print("Successfully connected to BigQuery!")

Starting Google Authentication...
Successfully connected to BigQuery!


In [15]:
query_total = """
SELECT
    safetyreportid,
    brand_queried,
    serious,
    seriousness_death,
    seriousness_lifethreatening,
    seriousness_hospitalization,
    seriousness_disabling,
    patient_age_yrs,
    patient_sex
FROM `glp-1-513513.glp_1_GI.glp_1_AE_2020_2024`
"""
df_total = client.query(query_total).to_dataframe()

query_gi = "SELECT * FROM `glp-1-513513.glp_1_GI.glp_1_AE_2020_2024_GI_Clean_Mapped`"
df_gi = client.query(query_gi).to_dataframe()

print("\n--- DATA LOADING COMPLETE ---")
print(f"Master Background Dataset Shape: {df_total.shape[0]:,} rows, {df_total.shape[1]} columns")
print(f"Clean GI-Isolated Dataset Shape: {df_gi.shape[0]:,} rows, {df_gi.shape[1]} columns")


--- DATA LOADING COMPLETE ---
Master Background Dataset Shape: 39,485 rows, 9 columns
Clean GI-Isolated Dataset Shape: 7,447 rows, 16 columns


In [16]:
total_unique_patients = df_total.groupby('brand_queried')['safetyreportid'].nunique().reset_index()
total_unique_patients.columns = ['brand_queried', 'total_unique_patients']

In [17]:
gi_unique_patients = df_gi.groupby('brand_queried')['safetyreportid'].nunique().reset_index()
gi_unique_patients.columns = ['brand_queried', 'gi_unique_patients']

In [18]:
gi_percent_df = pd.merge(gi_unique_patients, total_unique_patients, on='brand_queried')
gi_percent_df['gi_patient_percentage'] = (gi_percent_df['gi_unique_patients'] / gi_percent_df['total_unique_patients']) * 100
gi_percent_df['gi_patient_percentage'] = gi_percent_df['gi_patient_percentage'].round(2)

In [19]:
gi_percent_df.to_csv('gi_patient_rates.csv', index=False)

print(gi_percent_df)

  brand_queried  gi_unique_patients  total_unique_patients  \
0      MOUNJARO                1272                   4956   
1       OZEMPIC                 935                   2236   
2       SAXENDA                 575                   1755   
3        WEGOVY                1571                   4992   

   gi_patient_percentage  
0                  25.67  
1                  41.82  
2                  32.76  
3                  31.47  


In [20]:
gi_brand_matrix = pd.crosstab(
    index=df_gi['Clean_GI_sx'],
    columns=df_gi['brand_queried'],
    normalize='index'
)

gi_brand_df = (gi_brand_matrix * 100).round(2)
gi_brand_df = gi_brand_df.reset_index()

gi_brand_df.to_csv('brand_contribution.csv', index=False)

print(gi_brand_df)

brand_queried             Clean_GI_sx  MOUNJARO  OZEMPIC  SAXENDA  WEGOVY
0                      Abdominal Pain     28.82    25.66    12.54   32.97
1                    Biliary Disorder     10.67     9.49    39.92   39.92
2                        Constipation     25.09    25.26    13.57   36.08
3                         Dehydration     22.30    17.10    18.96   41.64
4                           Diarrhoea     32.54    27.42     8.49   31.56
5                                GERD     31.69    13.17    10.70   44.44
6                       Gastroparesis     17.74    17.74     6.45   58.06
7              Intestinal Obstruction     13.16    11.84    14.47   60.53
8                     Nausea/Vomiting     29.45    20.04    10.57   39.93
9                        Pancreatitis     13.42    35.62    21.64   29.32


In [21]:
sex_gi = df_gi[df_gi['patient_sex'] != 'Unknown'].copy()
sex_gi_counts = sex_gi.groupby(['brand_queried', 'patient_sex'])['safetyreportid'].nunique().reset_index(name='unique_patient_count')
sex_gi_counts['cohort_type'] = 'GI Side Effects Only'

sex_total = df_total[df_total['patient_sex'] != 'Unknown'].copy()
sex_total_counts = sex_total.groupby(['brand_queried', 'patient_sex'])['safetyreportid'].nunique().reset_index(name='unique_patient_count')
sex_total_counts['cohort_type'] = 'Overall System Baseline'

overall_sex_comparison = pd.concat([sex_gi_counts, sex_total_counts], ignore_index=True)

overall_sex_comparison.to_csv('overall_sex_comparison.csv', index=False)

print(overall_sex_comparison)

   brand_queried patient_sex  unique_patient_count              cohort_type
0       MOUNJARO      Female                   816     GI Side Effects Only
1       MOUNJARO        Male                   239     GI Side Effects Only
2        OZEMPIC      Female                   547     GI Side Effects Only
3        OZEMPIC        Male                   373     GI Side Effects Only
4        SAXENDA      Female                   480     GI Side Effects Only
5        SAXENDA        Male                    78     GI Side Effects Only
6         WEGOVY      Female                  1137     GI Side Effects Only
7         WEGOVY        Male                   250     GI Side Effects Only
8       MOUNJARO      Female                  3520  Overall System Baseline
9       MOUNJARO        Male                   829  Overall System Baseline
10       OZEMPIC      Female                  1271  Overall System Baseline
11       OZEMPIC        Male                   900  Overall System Baseline
12       SAX

In [22]:
gi_clean = df_gi[df_gi['patient_sex'] != 'Unknown']
gi_sex_counts = gi_clean.groupby(['brand_queried', 'patient_sex'])['safetyreportid'].nunique().to_frame(name='gi_unique_patients')

total_clean = df_total[df_total['patient_sex'] != 'Unknown']
total_sex_counts = total_clean.groupby(['brand_queried', 'patient_sex'])['safetyreportid'].nunique().to_frame(name='total_baseline_patients')

overall_sex_pct = gi_sex_counts.join(total_sex_counts, how='inner')

overall_sex_pct['gi_incidence_rate_pct'] = (overall_sex_pct['gi_unique_patients'] / overall_sex_pct['total_baseline_patients']) * 100
overall_sex_pct['gi_incidence_rate_pct'] = overall_sex_pct['gi_incidence_rate_pct'].round(2)

overall_sex_pct = overall_sex_pct.reset_index()
overall_sex_pct.to_csv('overall_sex_pct.csv', index=False)

print(overall_sex_pct)

  brand_queried patient_sex  gi_unique_patients  total_baseline_patients  \
0      MOUNJARO      Female                 816                     3520   
1      MOUNJARO        Male                 239                      829   
2       OZEMPIC      Female                 547                     1271   
3       OZEMPIC        Male                 373                      900   
4       SAXENDA      Female                 480                     1437   
5       SAXENDA        Male                  78                      269   
6        WEGOVY      Female                1137                     3725   
7        WEGOVY        Male                 250                      807   

   gi_incidence_rate_pct  
0                  23.18  
1                  28.83  
2                  43.04  
3                  41.44  
4                  33.40  
5                  29.00  
6                  30.52  
7                  30.98  


In [23]:
age_bins = [17, 34, 49, 64, 79, 120]
age_labels = ['18-34', '35-49', '50-64', '65-79', '80+']

age_gi = df_gi[df_gi['patient_age_yrs'].notnull() & (df_gi['patient_age_yrs'] > 17)].copy()
age_gi['age_bracket'] = pd.cut(age_gi['patient_age_yrs'], bins=age_bins, labels=age_labels)
gi_age_counts = age_gi.groupby(['brand_queried', 'age_bracket'], observed=False)['safetyreportid'].nunique().to_frame(name='gi_unique_patients')

age_total = df_total[df_total['patient_age_yrs'].notnull() & (df_total['patient_age_yrs'] > 17)].copy()
age_total['age_bracket'] = pd.cut(age_total['patient_age_yrs'], bins=age_bins, labels=age_labels)
total_age_counts = age_total.groupby(['brand_queried', 'age_bracket'], observed=False)['safetyreportid'].nunique().to_frame(name='total_baseline_patients')

overall_age_pct = gi_age_counts.join(total_age_counts, how='inner')

overall_age_pct['gi_incidence_rate_pct'] = (overall_age_pct['gi_unique_patients'] / overall_age_pct['total_baseline_patients']) * 100
overall_age_pct['gi_incidence_rate_pct'] = overall_age_pct['gi_incidence_rate_pct'].round(2)

overall_age_pct = overall_age_pct.reset_index()
overall_age_pct.to_csv('overall_age_percentages.csv', index=False)

print(overall_age_pct)

   brand_queried age_bracket  gi_unique_patients  total_baseline_patients  \
0       MOUNJARO       18-34                  81                      434   
1       MOUNJARO       35-49                 234                     1372   
2       MOUNJARO       50-64                 314                     1311   
3       MOUNJARO       65-79                  75                      269   
4       MOUNJARO         80+                   1                       16   
5        OZEMPIC       18-34                  14                       32   
6        OZEMPIC       35-49                  76                      162   
7        OZEMPIC       50-64                 252                      532   
8        OZEMPIC       65-79                 254                      573   
9        OZEMPIC         80+                  20                       53   
10       SAXENDA       18-34                  51                      122   
11       SAXENDA       35-49                 129                      331   

In [24]:
gi_dem_clean = df_gi[
    (df_gi['patient_sex'] != 'Unknown') &
    (df_gi['patient_age_yrs'].notnull()) &
    (df_gi['patient_age_yrs'] > 17)
].copy()

age_bins = [17, 34, 49, 64, 79, 120]
age_labels = ['18-34', '35-49', '50-64', '65-79', '80+']
gi_dem_clean['age_bracket'] = pd.cut(gi_dem_clean['patient_age_yrs'], bins=age_bins, labels=age_labels)

matrix_counts = gi_dem_clean.groupby(
    ['brand_queried', 'patient_sex', 'age_bracket', 'Clean_GI_sx'],
    observed=False
)['safetyreportid'].nunique().reset_index(name='patient_count')

cohort_totals = gi_dem_clean.groupby(
    ['brand_queried', 'patient_sex', 'age_bracket'],
    observed=False
)['safetyreportid'].nunique().reset_index(name='cohort_total_patients')


final_matrix = pd.merge(matrix_counts, cohort_totals, on=['brand_queried', 'patient_sex', 'age_bracket'])


final_matrix['symptom_share_pct'] = (final_matrix['patient_count'] / final_matrix['cohort_total_patients']) * 100
final_matrix['symptom_share_pct'] = final_matrix['symptom_share_pct'].round(2)


final_matrix.to_csv('gi_symptom_demographic_matrix.csv', index=False)

print(final_matrix.head(20))

   brand_queried patient_sex age_bracket             Clean_GI_sx  \
0       MOUNJARO      Female       18-34          Abdominal Pain   
1       MOUNJARO      Female       18-34        Biliary Disorder   
2       MOUNJARO      Female       18-34            Constipation   
3       MOUNJARO      Female       18-34             Dehydration   
4       MOUNJARO      Female       18-34               Diarrhoea   
5       MOUNJARO      Female       18-34                    GERD   
6       MOUNJARO      Female       18-34           Gastroparesis   
7       MOUNJARO      Female       18-34  Intestinal Obstruction   
8       MOUNJARO      Female       18-34         Nausea/Vomiting   
9       MOUNJARO      Female       18-34            Pancreatitis   
10      MOUNJARO      Female       35-49          Abdominal Pain   
11      MOUNJARO      Female       35-49        Biliary Disorder   
12      MOUNJARO      Female       35-49            Constipation   
13      MOUNJARO      Female       35-49        

In [25]:
patient_severity = df_gi.groupby(['brand_queried', 'Clean_GI_sx', 'safetyreportid'])[[
    'serious',
    'seriousness_hospitalization',
    'seriousness_death',
    'seriousness_lifethreatening',
    'seriousness_disabling'
]].max().reset_index()

# Convert boolean to numeric floats by taking the mean and multiplying by 100
# This calculates percentage incidence rate for each clinical endpoint per symptom per drug
endpoints_df = patient_severity.groupby(['brand_queried', 'Clean_GI_sx'])[[
    'serious',
    'seriousness_hospitalization',
    'seriousness_death',
    'seriousness_lifethreatening',
    'seriousness_disabling'
]].mean() * 100

endpoints_df = endpoints_df.round(2).reset_index()

endpoints_df.columns = [
    'brand_queried',
    'Clean_GI_sx',
    'any_serious_outcome_pct',
    'hospitalization_rate_pct',
    'death_rate_pct',
    'life_threatening_rate_pct',
    'disabling_outcome_pct'
]


endpoints_df.to_csv('endpoints.csv', index=False)

print(endpoints_df.head(10))

  brand_queried             Clean_GI_sx  any_serious_outcome_pct  \
0      MOUNJARO          Abdominal Pain                    15.51   
1      MOUNJARO        Biliary Disorder                    69.23   
2      MOUNJARO            Constipation                     13.7   
3      MOUNJARO             Dehydration                    48.33   
4      MOUNJARO               Diarrhoea                    20.07   
5      MOUNJARO                    GERD                    11.69   
6      MOUNJARO           Gastroparesis                    18.18   
7      MOUNJARO  Intestinal Obstruction                    100.0   
8      MOUNJARO         Nausea/Vomiting                     10.8   
9      MOUNJARO            Pancreatitis                    100.0   

   hospitalization_rate_pct  death_rate_pct  life_threatening_rate_pct  \
0                      7.76             0.0                        0.0   
1                     26.92             0.0                        0.0   
2                      4.79  

In [26]:
df_gi['receive_date'] = pd.to_datetime(df_gi['receive_date'])

df_gi['year_month'] = df_gi['receive_date'].dt.strftime('%Y-%m')

trends_df = df_gi.groupby(['year_month', 'brand_queried', 'Clean_GI_sx'])['safetyreportid'].nunique().reset_index()

trends_df.columns = ['year_month', 'brand_queried', 'Clean_GI_sx', 'unique_monthly_patients_count']

trends_df.to_csv('timeline_trends.csv', index=False)

print(trends_df.head(20))

   year_month brand_queried             Clean_GI_sx  \
0     2020-01       OZEMPIC          Abdominal Pain   
1     2020-01       OZEMPIC        Biliary Disorder   
2     2020-01       OZEMPIC            Constipation   
3     2020-01       OZEMPIC             Dehydration   
4     2020-01       OZEMPIC               Diarrhoea   
5     2020-01       OZEMPIC                    GERD   
6     2020-01       OZEMPIC           Gastroparesis   
7     2020-01       OZEMPIC  Intestinal Obstruction   
8     2020-01       OZEMPIC         Nausea/Vomiting   
9     2020-01       OZEMPIC            Pancreatitis   
10    2020-01       SAXENDA          Abdominal Pain   
11    2020-01       SAXENDA        Biliary Disorder   
12    2020-01       SAXENDA         Nausea/Vomiting   
13    2020-01        WEGOVY            Constipation   
14    2020-01        WEGOVY             Dehydration   
15    2020-01        WEGOVY               Diarrhoea   
16    2020-01        WEGOVY                    GERD   
17    2020